## Automatic annotation using LocateAnything text-prompt model

In [ ]:
import json
import sys
from pathlib import Path
from PIL import Image
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Also add src/ for locateanything_worker import
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
from locateanything_worker import LocateAnythingWorker

WEIGHTS_PATH = str(PROJECT_ROOT / "weights" / "locany" / "Locateanything-3B")

worker = LocateAnythingWorker(WEIGHTS_PATH)
print("✅ LocateAnything model loaded")

### Configure class → text prompt mapping

In [ ]:
# class_id → (Chinese label, English text prompt for LocateAnything)
class_config = {
    0: ("红盒子", "red box"),
    1: ("小物体", "small object"),
    2: ("绿色小盒子", "green small box"),
    3: ("玩具车", "toy car"),
}

target_folder = PROJECT_ROOT / "datasets/giftbox_task/images"
annotation_save_path = PROJECT_ROOT / "datasets/giftbox_task/annotations/annotations.json"

target_paths = sorted(target_folder.glob("*"))
print(f"Found {len(target_paths)} target images")
print(f"Classes: {[v[0] for v in class_config.values()]}")

## Run LocateAnything text-prompt inference & save COCO annotations

In [ ]:
import re


def bbox_to_polygon(x1, y1, x2, y2):
    """Convert bbox [x1,y1,x2,y2] to a rectangle polygon for COCO segmentation."""
    return [x1, y1, x2, y1, x2, y2, x1, y2]


def locany_predict_bboxes(worker, image, prompts):
    """
    Run LocateAny detect on a single image, return parsed bboxes.
    Returns list of (label_name, x1, y1, x2, y2).
    """
    result = worker.detect(image, prompts)
    answer = result["answer"]
    w, h = image.size

    # Parse both label + box coordinates from answer
    pattern = r"<ref>(.*?)</ref><box><(\d+)><(\d+)><(\d+)><(\d+)></box>"
    matches = re.findall(pattern, answer)

    results = []
    for label, x1_raw, y1_raw, x2_raw, y2_raw in matches:
        x1 = float(x1_raw) / 1000 * w
        y1 = float(y1_raw) / 1000 * h
        x2 = float(x2_raw) / 1000 * w
        y2 = float(y2_raw) / 1000 * h
        results.append((label, x1, y1, x2, y2))

    return results


def make_coco_annotation_locany(target_paths, class_config, worker, save_path, min_area=0):
    """
    Run LocateAny on all images, one detect call per image, save COCO JSON.
    """
    coco_output = {
        "images": [],
        "annotations": [],
        "categories": [
            {"id": cid, "name": label, "supercategory": "object"}
            for cid, (label, _) in class_config.items()
        ],
    }

    # Build label → class_id reverse mapping
    label_to_cid = {prompt: cid for cid, (label, prompt) in class_config.items()}
    all_prompts = [prompt for label, prompt in class_config.values()]

    ann_id = 1
    total_objs = 0

    # Build images list
    for img_id, target_path in enumerate(target_paths, start=1):
        target_img = Image.open(target_path).convert("RGB")
        w, h = target_img.size
        coco_output["images"].append({
            "id": img_id,
            "file_name": target_path.name,
            "width": w,
            "height": h,
        })

    # Run inference: per image, one call with all prompts
    for img_id, target_path in enumerate(tqdm(target_paths, desc="LocateAny inference"), start=1):
        image = Image.open(target_path).convert("RGB")
        bboxes = locany_predict_bboxes(worker, image, all_prompts)

        for label_name, x1, y1, x2, y2 in bboxes:
            bbox_w = x2 - x1
            bbox_h = y2 - y1
            area = bbox_w * bbox_h
            if area < min_area:
                continue
            if label_name not in label_to_cid:
                continue

            seg = bbox_to_polygon(x1, y1, x2, y2)
            coco_output["annotations"].append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": label_to_cid[label_name],
                "segmentation": [seg],
                "bbox": [float(x1), float(y1), float(bbox_w), float(bbox_h)],
                "area": float(area),
                "iscrowd": 0,
            })
            ann_id += 1
            total_objs += 1

    # Save JSON
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(coco_output, f, indent=2, ensure_ascii=False)

    # Save labels.txt
    labels_path = save_path.parent / "labels.txt"
    with open(labels_path, "w", encoding="utf-8") as f_labels:
        for cid in sorted(class_config.keys()):
            f_labels.write(f"{class_config[cid][0]}\n")

    print(f"\n✅ COCO annotation saved to {save_path}")
    print(f"   images={len(coco_output['images'])}, "
          f"annotations={len(coco_output['annotations'])}, "
          f"categories={len(coco_output['categories'])}")
    print(f"   total objects found: {total_objs}")

In [ ]:
# Run auto-annotation
if not annotation_save_path.exists():
    annotation_save_path.parent.mkdir(parents=True, exist_ok=True)
    annotation_save_path.touch()

make_coco_annotation_locany(
    target_paths=target_paths,
    class_config=class_config,
    worker=worker,
    save_path=annotation_save_path,
    min_area=0,
)